# **Limpeza e padronização — ENEM 2025**

- **Objetivo:** adequar os tipos de dados, padronizar as colunas textuais e preservar corretamente as categorias e ausências identificadas durante a inspeção.
- **Processo:** registro das métricas iniciais, adequação dos tipos, padronização textual, validação antes e depois das transformações e exportação da base limpa.

In [44]:
## Carregamento das bibliotecas e criação dos caminhos de diretórios

import os
from pathlib import Path
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import pyarrow.parquet as pq
pd.options.display.max_columns = None
pd.options.display.float_format = "{:.3f}".format

load_dotenv()

DATA_DIR = Path(os.environ["ENEM_RAW_DIR"]) / "DADOS"
PARTICIPANTS_FILE = DATA_DIR / "PARTICIPANTES_2025.csv"

In [45]:
participantes_clean = pd.read_csv( PARTICIPANTS_FILE, sep=";" , encoding="latin-1")
participantes_antes = participantes_clean.copy()

In [46]:
# 1 - Registro das métricas de referência antes das transformações
perfil_antes = pd.DataFrame(
    {
        "tipo_antes": participantes_clean.dtypes.astype(str),
        "nulos_antes": participantes_clean.isna().sum(),
        "valores_unicos_antes": participantes_clean.nunique(),
        "valores_duplicados_antes": participantes_clean.duplicated().sum()
    }
)

display(perfil_antes)

,tipo_antes,nulos_antes,valores_unicos_antes,valores_duplicados_antes
NU_INSCRICAO,int64,0,4810772,0
NU_ANO,int64,0,1,0
TP_FAIXA_ETARIA,int64,0,20,0
TP_SEXO,str,0,2,0
TP_ESTADO_CIVIL,int64,0,5,0
TP_COR_RACA,int64,0,6,0
TP_NACIONALIDADE,int64,0,5,0
TP_ST_CONCLUSAO,int64,0,4,0
TP_ANO_CONCLUIU,int64,0,20,0
TP_ENSINO,float64,3080608,2,0


In [47]:
## 2 - alteração dos tipos de dados


tipos_esperados = {
    "NU_INSCRICAO": "string",
    "TP_ENSINO": "Int8",
    "CO_MUNICIPIO_PROVA": "string",
    "CO_UF_PROVA": "string",
    "Q005": "int",
}

participantes_clean = participantes_clean.astype(tipos_esperados)

comparacao = pd.DataFrame({
    "antes" : perfil_antes["tipo_antes"],
    "depois" : participantes_clean.dtypes
})

display(comparacao)

## Alteração dos tipos de dados conforme verificação na etapa de inspeção

,antes,depois
NU_INSCRICAO,int64,string
NU_ANO,int64,int64
TP_FAIXA_ETARIA,int64,int64
TP_SEXO,str,str
TP_ESTADO_CIVIL,int64,int64
TP_COR_RACA,int64,int64
TP_NACIONALIDADE,int64,int64
TP_ST_CONCLUSAO,int64,int64
TP_ANO_CONCLUIU,int64,int64
TP_ENSINO,float64,Int8


In [48]:

## 3 - padronização das colunas texto

colunas_texto = [
    "NU_INSCRICAO",
    "CO_MUNICIPIO_PROVA",
    "CO_UF_PROVA",
    "NO_MUNICIPIO_PROVA"]

for coluna in colunas_texto:
    participantes_clean[coluna] = (participantes_clean[coluna].str.strip())


## Transformação apenas das colunas texto que poderiam conter espaços indevidos

In [49]:
 ## 4 - validação de nulos e duplicatas aós alterações
perfil_antes = pd.DataFrame({
        "tipo_antes": participantes_antes.dtypes.astype(str),
        "nulos_antes": participantes_antes.isna().sum(),
        "valores_unicos_antes": participantes_antes.nunique(),
        "valores_duplicados_antes": participantes_antes.duplicated().sum()
    })

perfil_depois = pd.DataFrame({
        "tipo_depois": participantes_clean.dtypes.astype(str),
        "nulos_depois": participantes_clean.isna().sum(),
        "valores_unicos_depois": participantes_clean.nunique(),
        "valores_duplicados_depois": participantes_clean.duplicated().sum()
    })

comparacao_perfil = pd.concat([perfil_antes, perfil_depois],axis=1)
display (comparacao_perfil)

## A quantidade de valores nulos e a ausência de duplicatas permaneceu inalterada

,tipo_antes,nulos_antes,valores_unicos_antes,valores_duplicados_antes,tipo_depois,nulos_depois,valores_unicos_depois,valores_duplicados_depois
NU_INSCRICAO,int64,0,4810772,0,string,0,4810772,0
NU_ANO,int64,0,1,0,int64,0,1,0
TP_FAIXA_ETARIA,int64,0,20,0,int64,0,20,0
TP_SEXO,str,0,2,0,str,0,2,0
TP_ESTADO_CIVIL,int64,0,5,0,int64,0,5,0
TP_COR_RACA,int64,0,6,0,int64,0,6,0
TP_NACIONALIDADE,int64,0,5,0,int64,0,5,0
TP_ST_CONCLUSAO,int64,0,4,0,int64,0,4,0
TP_ANO_CONCLUIU,int64,0,20,0,int64,0,20,0
TP_ENSINO,float64,3080608,2,0,Int8,3080608,2,0


In [50]:
comparacao_cardinalidade = pd.DataFrame({
    "valores_unicos_antes": perfil_antes["valores_unicos_antes"],
    "valores_unicos_depois": participantes_clean.nunique(),
})

comparacao_cardinalidade["diferenca"] = (
    comparacao_cardinalidade["valores_unicos_depois"]
    - comparacao_cardinalidade["valores_unicos_antes"]
    )

comparacao_cardinalidade

## A cardinalidade dos valores permaneceu inalterada após a limpeza

,valores_unicos_antes,valores_unicos_depois,diferenca
NU_INSCRICAO,4810772,4810772,0
NU_ANO,1,1,0
TP_FAIXA_ETARIA,20,20,0
TP_SEXO,2,2,0
TP_ESTADO_CIVIL,5,5,0
TP_COR_RACA,6,6,0
TP_NACIONALIDADE,5,5,0
TP_ST_CONCLUSAO,4,4,0
TP_ANO_CONCLUIU,20,20,0
TP_ENSINO,2,2,0


In [ ]:
# Salvamento dos dados em Parquet

diretorio_atual = Path.cwd()

PROJECT_ROOT = (
diretorio_atual.parent
    if diretorio_atual.name == "notebooks"
    else diretorio_atual
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_FILE = PROCESSED_DIR / "participantes_2025_limpo.parquet"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

participantes_clean.to_parquet(OUTPUT_FILE,engine="pyarrow",compression="snappy",index=False,)


# Base limpa e exportada. Esse formato preserva os tipos de dados, ofereceendo compressão e permitindo leituras mais eficientes do que o CSV original